In [4]:
#import and config
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import faiss

processed_path = "../dataset/processed"
models_path = "../Models"

retrieval_interactions = pd.read_parquet(
    os.path.join(processed_path, "retrieval_interactions.parquet")
)

movie_features = pd.read_parquet(
    os.path.join(processed_path, "movie_features.parquet")
)

user_features = pd.read_parquet(
    os.path.join(processed_path, "user_features.parquet")
)

print("FAISS version:", faiss.__version__)
print("Retrieval interactions:", retrieval_interactions.shape)
print("Movie features:", movie_features.shape)
print("User features:", user_features.shape)

FAISS version: 1.16.0
Retrieval interactions: (575281, 3)
Movie features: (3883, 20)
User features: (6040, 4)


In [5]:
#Loading the trained movie embeddings
movie_ids = sorted(
    retrieval_interactions["movie_id"].unique().tolist()
)

embedding_dim = 64

movie_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(embedding_dim,))
])

print("Movie vocabulary:", len(movie_ids))


Movie vocabulary: 3533


In [6]:
#recreating the exact training split
retrieval_sorted = retrieval_interactions.sort_values(
    ["user_id", "timestamp"]
).copy()

retrieval_sorted["position"] = (
    retrieval_sorted.groupby("user_id").cumcount()
)

retrieval_sorted["user_count"] = (
    retrieval_sorted.groupby("user_id")["user_id"].transform("size")
)

retrieval_sorted["test_count"] = np.where(
    retrieval_sorted["user_count"] >= 2,
    np.maximum(1, np.ceil(retrieval_sorted["user_count"] * 0.2)),
    0
)

train_df = retrieval_sorted[
    retrieval_sorted["position"] < (
        retrieval_sorted["user_count"] - retrieval_sorted["test_count"]
    )
].copy()

test_df = retrieval_sorted[
    retrieval_sorted["position"] >= (
        retrieval_sorted["user_count"] - retrieval_sorted["test_count"]
    )
].copy()

movie_ids = sorted(train_df["movie_id"].unique().tolist())

print("Training movies:", len(movie_ids))
print("Training interactions:", len(train_df))
print("Testing interactions:", len(test_df))

Training movies: 3456
Training interactions: 457819
Testing interactions: 117462


In [7]:
#movie tower re defining 
class MovieModel(tf.keras.Model):
    def __init__(self, movie_ids):
        super().__init__()

        self.movie_lookup = tf.keras.layers.StringLookup(
            vocabulary=movie_ids,
            mask_token=None
        )

        self.movie_embedding = tf.keras.layers.Embedding(
            input_dim=len(movie_ids) + 1,
            output_dim=64
        )

    def call(self, movie_id):
        movie_index = self.movie_lookup(movie_id)
        return self.movie_embedding(movie_index)


print("Movie model architecture defined.")

Movie model architecture defined.


In [8]:
#Movie Tower and restore trained weights
movie_model = MovieModel(movie_ids)

movie_model(tf.constant(movie_ids[:1]))

movie_model.load_weights(
    "../Models/movie_model.weights.h5"
)

print("Movie model weights loaded successfully.")


Movie model weights loaded successfully.


In [9]:
#generating movie embedding
movie_id_tensor = tf.constant(movie_ids)

movie_embeddings = movie_model(movie_id_tensor).numpy()

print("Movie embeddings shape:", movie_embeddings.shape)
print("Embedding data type:", movie_embeddings.dtype)

Movie embeddings shape: (3456, 64)
Embedding data type: float32


In [10]:
#creating the FAISS index
movie_embeddings = np.ascontiguousarray(
    movie_embeddings, dtype=np.float32
)

embedding_dim = movie_embeddings.shape[1]

movie_index = faiss.IndexFlatIP(embedding_dim)
movie_index.add(movie_embeddings)

print("FAISS index size:", movie_index.ntotal)
print("Embedding dimension:", embedding_dim)

FAISS index size: 3456
Embedding dimension: 64


In [11]:
#loading traning user tower
user_ids = sorted(train_df["user_id"].unique().tolist())

class UserModel(tf.keras.Model):
    def __init__(self, user_ids):
        super().__init__()

        self.user_lookup = tf.keras.layers.StringLookup(
            vocabulary=user_ids,
            mask_token=None
        )

        self.user_embedding = tf.keras.layers.Embedding(
            input_dim=len(user_ids) + 1,
            output_dim=64
        )

    def call(self, user_id):
        user_index = self.user_lookup(user_id)
        return self.user_embedding(user_index)

print("User model architecture defined.")
print("Training users:", len(user_ids))

User model architecture defined.
Training users: 6038


In [12]:
#initializing the user tower and loading the trained weight 
user_model = UserModel(user_ids)

user_model(tf.constant(user_ids[:1]))

user_model.load_weights(
    "../Models/user_model.weights.h5"
)

print("User model weights loaded successfully.")

User model weights loaded successfully.


In [13]:
user_id_tensor = tf.constant(user_ids)

user_embeddings = user_model(user_id_tensor).numpy()

print("User embeddings shape:", user_embeddings.shape)
print("Embedding data type:", user_embeddings.dtype)

User embeddings shape: (6038, 64)
Embedding data type: float32


In [14]:
#mapping FAISS result index
movie_id_mapping = np.array(movie_ids)

print("Mapping size:", len(movie_id_mapping))
print("First 5 movie IDs:", movie_id_mapping[:5])

Mapping size: 3456
First 5 movie IDs: ['1' '10' '100' '1000' '1002']


In [15]:
#saving the faiss movie index and id mapping 
faiss.write_index(
    movie_index,
    os.path.join(models_path, "movie_faiss.index")
)

np.save(
    os.path.join(models_path, "movie_ids.npy"),
    movie_id_mapping
)

print("FAISS index saved successfully.")
print("Movie ID mapping saved successfully.")

FAISS index saved successfully.
Movie ID mapping saved successfully.


In [17]:
#testing fasiss index
query_embedding = user_embeddings[0].reshape(1, -1)

distances, indices = movie_index.search(query_embedding, 10)

recommended_movie_ids = movie_id_mapping[indices[0]]

print("Recommended movie IDs:", recommended_movie_ids)
print("Similarity scores:", distances[0])

Recommended movie IDs: ['914' '1947' '2080' '919' '920' '1028' '595' '1022' '17' '899']
Similarity scores: [1.1950114  1.0429299  1.0230067  0.99978924 0.9989766  0.99040544
 0.9875762  0.97369236 0.96137416 0.9518163 ]


In [23]:
#retrieving the top 10 movies 
candidate_scores, candidate_indices = movie_index.search(
    np.ascontiguousarray(user_embeddings, dtype=np.float32),
    200
)

print("Candidate indices shape:", candidate_indices.shape)
print("Candidate scores shape:", candidate_scores.shape)

Candidate indices shape: (6038, 200)
Candidate scores shape: (6038, 200)


In [24]:
user_recommendations = {}

for user_idx, user_id in enumerate(user_ids):
    watched = set(
        train_df.loc[
            train_df["user_id"] == user_id,
            "movie_id"
        ]
    )

    recommendations = []

    for score, movie_idx in zip(
        candidate_scores[user_idx],
        candidate_indices[user_idx]
    ):
        if movie_idx == -1:
            continue

        movie_id = movie_id_mapping[movie_idx]

        if movie_id not in watched:
            recommendations.append((movie_id, float(score)))

        if len(recommendations) == 10:
            break

    user_recommendations[user_id] = recommendations

print("Users processed:", len(user_recommendations))
print("Recommendations for first user:", len(user_recommendations[user_ids[0]]))
print("First 10 recommendations:", user_recommendations[user_ids[0]])

Users processed: 6038
Recommendations for first user: 10
First 10 recommendations: [(np.str_('914'), 1.1950114965438843), (np.str_('1947'), 1.0429301261901855), (np.str_('2080'), 1.023006796836853), (np.str_('920'), 0.998976469039917), (np.str_('595'), 0.9875761866569519), (np.str_('17'), 0.9613742232322693), (np.str_('899'), 0.9518161416053772), (np.str_('2565'), 0.8965814113616943), (np.str_('2081'), 0.8944977521896362), (np.str_('953'), 0.8823740482330322)]


In [26]:
#saving the generated recommendation result
recommendation_rows = []

for user_id, recommendations in user_recommendations.items():
    for rank, (movie_id, score) in enumerate(recommendations, start=1):
        recommendation_rows.append({
            "user_id": user_id,
            "movie_id": movie_id,
            "retrieval_score": score,
            "rank": rank
        })

recommendations_df = pd.DataFrame(recommendation_rows)

recommendations_df.to_parquet(
    os.path.join(processed_path, "faiss_recommendations.parquet"),
    index=False
)

print("Recommendations saved:", recommendations_df.shape)
print(recommendations_df.head())

Recommendations saved: (60380, 4)
  user_id movie_id  retrieval_score  rank
0       1      914         1.195011     1
1       1     1947         1.042930     2
2       1     2080         1.023007     3
3       1      920         0.998976     4
4       1      595         0.987576     5
